# power_brick — Dual-Rail 12V → 5V / 3.3V Power Supply Brick

12V barrel/screw input → Buck #1 (TPS54302, 5V@2A) + Buck #2 (TPS54302, 3.3V@1A). Both rails exposed via 2-pin output headers.

In [1]:
import hw_toolkit as hw
from hw_toolkit.parts import Buck
import pathlib

board = hw.Board("power_brick")
board

Board(project_id='power_brick', subsystems=0, interfaces=0)

In [2]:
# 12V input connector (2-pin screw/barrel terminal)
# Connector_Generic:Conn_01x02 resolves to a real KiCad symbol.
# Pin_1 = VIN (12V), Pin_2 = GND
j_in = board.module(
    id="j_in",
    category="connector",
    mpn="Conn_01x02",
    package="PinHeader_2.54mm_1x02",
    manufacturer="Generic",
    price_usd=0.10,
)
j_in

Module(pick=SubsystemPick(id='j_in', category='connector', mpn='Conn_01x02', manufacturer='Generic', lcsc=None, package='PinHeader_2.54mm_1x02', datasheet_url='', lib_id='Connector_Generic:Conn_01x02', footprint=None, group=None, qty_per_board=1, price_usd=0.1, stock=0, actuals={}, port_bindings={}), board=Board(project_id='power_brick', subsystems=1, interfaces=0), math=None, notes=[])

In [3]:
# Buck #1: 12V -> 5V @ 2A
# TPS54302 Vref = 0.8V: Vout = Vref * (1 + Rtop/Rbot)
# For 5V: Rtop/Rbot = (5.0/0.8) - 1 = 5.25
# Rtop=52.3k, Rbot=10k => Vout = 0.8 * (1 + 52.3/10) = 0.8 * 6.23 = 4.984V ≈ 5V
buck_5v = Buck(
    board,
    id="buck_5v",
    mpn="TPS54302",
    package="SOT-23-6",
    vin=12.0,
    vout=5.0,
    l="10uH",
    cin="10uF",
    cout="22uF",
    cboot="100nF",
    rtop="52.3k",
    rbot="10k",
    cap_package="0805",
    res_package="0603",
    ind_package="1210",
)
buck_5v

Module(pick=SubsystemPick(id='buck_5v', category='buck_converter', mpn='TPS54302', manufacturer='', lcsc=None, package='SOT-23-6', datasheet_url='', lib_id='Regulator_Switching:TPS54302', footprint='Package_TO_SOT_SMD:SOT-23-6', group='buck_5v', qty_per_board=1, price_usd=0.0, stock=0, actuals={}, port_bindings={}), board=Board(project_id='power_brick', subsystems=8, interfaces=0), math=None, notes=[])

In [4]:
# Buck #2: 12V -> 3.3V @ 1A
# For 3.3V: Rtop/Rbot = (3.3/0.8) - 1 = 3.125
# Rtop=31.6k, Rbot=10k => Vout = 0.8 * (1 + 31.6/10) = 0.8 * 4.16 = 3.328V ≈ 3.3V
buck_3v3 = Buck(
    board,
    id="buck_3v3",
    mpn="TPS54302",
    package="SOT-23-6",
    vin=12.0,
    vout=3.3,
    l="10uH",
    cin="10uF",
    cout="22uF",
    cboot="100nF",
    rtop="31.6k",
    rbot="10k",
    cap_package="0805",
    res_package="0603",
    ind_package="1210",
)
buck_3v3

Module(pick=SubsystemPick(id='buck_3v3', category='buck_converter', mpn='TPS54302', manufacturer='', lcsc=None, package='SOT-23-6', datasheet_url='', lib_id='Regulator_Switching:TPS54302', footprint='Package_TO_SOT_SMD:SOT-23-6', group='buck_3v3', qty_per_board=1, price_usd=0.0, stock=0, actuals={}, port_bindings={}), board=Board(project_id='power_brick', subsystems=15, interfaces=0), math=None, notes=[])

In [5]:
# 5V output header
j_5v = board.module(
    id="j_5v",
    category="connector",
    mpn="Conn_01x02",
    package="PinHeader_2.54mm_1x02",
    manufacturer="Generic",
    price_usd=0.10,
)
j_5v

Module(pick=SubsystemPick(id='j_5v', category='connector', mpn='Conn_01x02', manufacturer='Generic', lcsc=None, package='PinHeader_2.54mm_1x02', datasheet_url='', lib_id='Connector_Generic:Conn_01x02', footprint=None, group=None, qty_per_board=1, price_usd=0.1, stock=0, actuals={}, port_bindings={}), board=Board(project_id='power_brick', subsystems=16, interfaces=0), math=None, notes=[])

In [6]:
# 3.3V output header
j_3v3 = board.module(
    id="j_3v3",
    category="connector",
    mpn="Conn_01x02",
    package="PinHeader_2.54mm_1x02",
    manufacturer="Generic",
    price_usd=0.10,
)
j_3v3

Module(pick=SubsystemPick(id='j_3v3', category='connector', mpn='Conn_01x02', manufacturer='Generic', lcsc=None, package='PinHeader_2.54mm_1x02', datasheet_url='', lib_id='Connector_Generic:Conn_01x02', footprint=None, group=None, qty_per_board=1, price_usd=0.1, stock=0, actuals={}, port_bindings={}), board=Board(project_id='power_brick', subsystems=17, interfaces=0), math=None, notes=[])

In [7]:
# --- Net wiring ---
#
# 12V input rail:
# Buck factory creates per-buck VIN nets (buck_5v_vin, buck_3v3_vin).
# We stitch both onto the single 12V node by extending buck_5v_vin
# with the input connector and all of buck_3v3's VIN-side pins.
# (KiCad netlist reconciliation handles the pin appearing in both nets
# by ultimately treating them as one net.)
board.nets["buck_5v_vin"] += (
    "j_in.Pin_1",         # connector 12V positive
    "buck_3v3.VIN",       # 3.3V buck IC VIN pin
    "buck_3v3_cin.1",     # 3.3V buck input cap positive
    "buck_3v3.EN",        # 3.3V buck EN (always-on tie)
)

# GND rail: shared ground net (created by first Buck call)
gnd = board.nets["gnd"]
gnd += "j_in.Pin_2", "j_5v.Pin_2", "j_3v3.Pin_2"

# 5V output: Buck factory created buck_5v_vout with the inductor output
board.nets["buck_5v_vout"] += "j_5v.Pin_1"

# 3.3V output: Buck factory created buck_3v3_vout
board.nets["buck_3v3_vout"] += "j_3v3.Pin_1"

print("12V rail:", board.nets["buck_5v_vin"])
print("GND rail:", board.nets["gnd"])
print("5V rail:", board.nets["buck_5v_vout"])
print("3.3V rail:", board.nets["buck_3v3_vout"])

12V rail: Net(id='buck_5v_vin', type='power', spec='12.0V', members=7)
GND rail: Net(id='gnd', type='power', spec='0V', members=11)
5V rail: Net(id='buck_5v_vout', type='power', spec='5.0V', members=4)
3.3V rail: Net(id='buck_3v3_vout', type='power', spec='3.3V', members=4)


In [8]:
board.summary()

"Board 'power_brick'\n  parts (17):\n    j_in           Conn_01x02                     PinHeader_2.54mm_1x02  $ 0.10\n    buck_5v        TPS54302                       SOT-23-6        $ 0.00\n    buck_5v_cin    C_10uF_0805                    0805            $ 0.02\n    buck_5v_cout   C_22uF_0805                    0805            $ 0.02\n    buck_5v_cboot  C_100nF_0402                   0402            $ 0.02\n    buck_5v_l      L_10uH_1210                    1210            $ 0.05\n    buck_5v_rtop   R_52.3k_0603                   0603            $ 0.01\n    buck_5v_rbot   R_10k_0603                     0603            $ 0.01\n    buck_3v3       TPS54302                       SOT-23-6        $ 0.00\n    buck_3v3_cin   C_10uF_0805                    0805            $ 0.02\n    buck_3v3_cout  C_22uF_0805                    0805            $ 0.02\n    buck_3v3_cboot C_100nF_0402                   0402            $ 0.02\n    buck_3v3_l     L_10uH_1210                    1210            $ 

In [9]:
# ERC — all parts (TPS54302, Device:R/C/L, Connector_Generic:Conn_01x02)
# resolve to real KiCad symbols, so gate on ERC_REAL_SYMBOL_CODES (tighter set).
board.check_erc(expected_codes=hw.ERC_REAL_SYMBOL_CODES)

In [10]:
# Export — absolute path per AGENT_GUIDE §7
out = pathlib.Path("/Users/juanantonioluera/ws/hw-toolkit/docs/projects/power_brick/power_brick.zip")
board.export_kicad(out, unzip=True, expected_codes=hw.ERC_REAL_SYMBOL_CODES)
print("Exported:", out)
assert out.exists(), "zip not created!"
print("ZIP size:", out.stat().st_size, "bytes")

Exported: /Users/juanantonioluera/ws/hw-toolkit/docs/projects/power_brick/power_brick.zip
ZIP size: 11969 bytes
